## **LLM Benchmarking**

### **Data Preparation**

Pull test data and class names from artifacts

In [1]:
import joblib

x_test = joblib.load(r".\Artifacts\x_test.pkl")
y_test = joblib.load(r".\Artifacts\y_test.pkl")
class_names = joblib.load(r".\Artifacts\class_names.pkl")

print("Data split loaded")

Data split loaded


In [5]:
x_test

['today tomorrow try move mountains cross fingers',
 'typa girl bullied lesbians high school',
 'cool woman gave directions polling place whose wee boy always runs around high heels ‘girls’ clothes he’s told gets bullied ‘poof’',
 'need teach version quran terrorists killing people world blast selves kill innocent isis terror org bearing enough bull shits cant buy version see jehad gazwa hind darulharb',
 'rome destroy coliseum roman forum every building statue romans slaves persecutedkilled christians support radical “muslims” iraq syria blew historic cultural sites',
 'try hide jihadi propaganda know hell gys specly jihadin aunty urgnt need hide islamic terrorisms behind typ artificial propaganda coz current scenario exposed infrnt whole world',
 'city college',
 'pepperidge farm remembers',
 'added feminazi white knight professional victims terminology',
 'blm something know muslim leaders record stating use manipulate goals quoted saying islam’s ‘useful idiots’ islam says that whit

In [6]:
y_test

array([4, 0, 0, ..., 3, 2, 3], shape=(11923,))

In [4]:
class_names

Index(['age', 'ethnicity', 'gender', 'not_cyberbullying',
       'other_cyberbullying', 'religion'],
      dtype='object')

### **LLM: Qwen2.5:7b-Instruct**

#### **Predictions**

In [15]:
import os
import json
import re
import time
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import joblib

# ----------------------------
# Ollama configuration
# ----------------------------
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:7b-instruct")  # e.g., deepseek-r1:7b, qwen2.5:7b-instruct
MAX_OLLAMA_CASES = None  # set integer (e.g., 200) for a smaller quick run
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 3

# Checkpoint/resume configuration
CHECKPOINT_EVERY = 25
RESUME_FROM_CHECKPOINT = True
RESET_CHECKPOINT = False

# If earlier cells were not run, load artifacts here as a fallback.
if "x_test" not in globals():
    x_test = joblib.load(r".\Artifacts\x_test.pkl")
if "y_test" not in globals():
    y_test = joblib.load(r".\Artifacts\y_test.pkl")
if "class_names" not in globals():
    class_names = joblib.load(r".\Artifacts\class_names.pkl")

class_names_list = list(class_names)
label_to_id = {label: i for i, label in enumerate(class_names_list)}
FALLBACK_LABEL = "other_cyberbullying"

if MAX_OLLAMA_CASES is None:
    x_eval = list(x_test)
    y_eval = np.asarray(y_test)
else:
    x_eval = list(x_test)[:MAX_OLLAMA_CASES]
    y_eval = np.asarray(y_test)[:MAX_OLLAMA_CASES]

artifacts_dir = Path(r".\Artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
safe_model_name = re.sub(r"[^a-zA-Z0-9_.-]", "_", OLLAMA_MODEL)
checkpoint_path = artifacts_dir / f"ollama_checkpoint_{safe_model_name}.pkl"
predictions_path = artifacts_dir / f"ollama_predictions_{safe_model_name}.pkl"

if RESET_CHECKPOINT and checkpoint_path.exists():
    checkpoint_path.unlink()
    print(f"Deleted old checkpoint: {checkpoint_path}")

SYSTEM_PROMPT = f"""You are an expert annotator for cyberbullying tweet classification.

Valid labels: {class_names_list}

Rules:
1) Return exactly one label from the valid label list.
2) Output STRICT JSON only, with this schema:
   {{"label": "<one_valid_label>"}}
3) Do not output markdown, code fences, or any extra keys.
"""

def _parse_json_payload(raw_text: str):
    text = (raw_text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE | re.DOTALL).strip()
    return json.loads(text)

def _normalize_text(s: str) -> str:
    return re.sub(r"[^a-z0-9_]+", "", (s or "").strip().lower())

normalized_label_to_canonical = {
    _normalize_text(label): label for label in class_names_list
}

def _map_to_valid_label(raw_label: str) -> str:
    # 1) Exact canonical label
    if raw_label in label_to_id:
        return raw_label

    raw_norm = _normalize_text(raw_label)

    # 2) Exact normalized match (handles spaces/case/punctuation variants)
    if raw_norm in normalized_label_to_canonical:
        return normalized_label_to_canonical[raw_norm]

    # 3) Substring match against any class label
    for cls in class_names_list:
        cls_norm = _normalize_text(cls)
        if cls_norm and cls_norm in raw_norm:
            return cls

    # 4) Fallback if no class detected
    if FALLBACK_LABEL in label_to_id:
        return FALLBACK_LABEL

    raise ValueError(
        f"Invalid label returned and fallback label not found: {raw_label}. "
        f"Expected one of {class_names_list} with fallback '{FALLBACK_LABEL}'."
    )

def ollama_predict_one(tweet_text: str):
    last_error = None
    payload = {
        "model": OLLAMA_MODEL,
        "stream": False,
        "format": "json",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Classify this tweet:\n\n{tweet_text}"},
        ],
        "options": {"temperature": 0},
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                f"{OLLAMA_BASE_URL}/api/chat",
                json=payload,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            resp.raise_for_status()
            data = resp.json()
            content = data.get("message", {}).get("content", "")
            out = _parse_json_payload(content)
            raw_label = str(out.get("label", "")).strip()
            final_label = _map_to_valid_label(raw_label)
            return label_to_id[final_label], final_label
        except Exception as err:
            last_error = err
            if attempt < MAX_RETRIES:
                time.sleep(1.2 * attempt)

    raise RuntimeError(f"Ollama prediction failed after retries: {last_error}")

def save_checkpoint(next_index: int, pred_ids, pred_rows):
    state = {
        "model": OLLAMA_MODEL,
        "x_eval_size": len(x_eval),
        "next_index": int(next_index),
        "pred_ids": list(pred_ids),
        "pred_rows": list(pred_rows),
        "saved_at_unix": time.time(),
    }
    joblib.dump(state, checkpoint_path)

# ----------------------------
# Resume from checkpoint if available
# ----------------------------
ollama_pred_ids = []
ollama_rows = []
start_index = 0

if RESUME_FROM_CHECKPOINT and checkpoint_path.exists():
    state = joblib.load(checkpoint_path)
    same_model = state.get("model") == OLLAMA_MODEL
    same_size = int(state.get("x_eval_size", -1)) == len(x_eval)
    if same_model and same_size:
        ollama_pred_ids = list(state.get("pred_ids", []))
        ollama_rows = list(state.get("pred_rows", []))
        start_index = int(state.get("next_index", len(ollama_pred_ids)))
        start_index = max(0, min(start_index, len(x_eval)))
        print(f"Resumed from checkpoint: {checkpoint_path}")
        print(f"Loaded {len(ollama_pred_ids)} saved predictions; continuing from index {start_index}")
    else:
        print("Checkpoint found but skipped due to model/test-size mismatch.")
        print(f"Checkpoint path: {checkpoint_path}")

# ----------------------------
# Run Ollama prediction for each tweet
# ----------------------------
total = len(x_eval)
start_time = time.time()

for idx in range(start_index, total):
    tweet_text = x_eval[idx]
    pred_id, pred_label = ollama_predict_one(str(tweet_text))
    ollama_pred_ids.append(int(pred_id))
    ollama_rows.append({
        "tweet": str(tweet_text),
        "y_test_label": class_names_list[int(y_eval[idx])],
        "ollama_prediction": pred_label,
    })

    processed = idx + 1
    if (processed % CHECKPOINT_EVERY == 0) or (processed == total):
        save_checkpoint(next_index=processed, pred_ids=ollama_pred_ids, pred_rows=ollama_rows)
        elapsed = time.time() - start_time
        done_now = max(1, processed - start_index)
        sec_per_tweet = elapsed / done_now
        remaining = total - processed
        eta_min = (remaining * sec_per_tweet) / 60.0
        print(f"Ollama progress: {processed}/{total} | checkpoint saved | ETA ~ {eta_min:.1f} min")

# Convert and save final artifacts
ollama_pred_ids = np.asarray(ollama_pred_ids, dtype=int)
ollama_pred_df = pd.DataFrame(ollama_rows)
ollama_pred_df["is_match"] = ollama_pred_df["y_test_label"] == ollama_pred_df["ollama_prediction"]

prediction_compare_df = ollama_pred_df[[
    "tweet",
    "y_test_label",
    "ollama_prediction",
    "is_match",
]].copy()

joblib.dump(
    {
        "ollama_pred_ids": ollama_pred_ids,
        "prediction_compare_df": prediction_compare_df,
        "model": OLLAMA_MODEL,
        "x_eval_size": len(x_eval),
    },
    predictions_path,
 )

display(prediction_compare_df.head(20))
print(f"Done. Collected {len(ollama_pred_ids)} predictions from model: {OLLAMA_MODEL}")
print(f"Total exact label matches: {int(prediction_compare_df['is_match'].sum())}/{len(prediction_compare_df)}")
print(f"Checkpoint file: {checkpoint_path}")
print(f"Predictions artifact: {predictions_path}")

Ollama progress: 25/11923 | checkpoint saved | ETA ~ 1063.2 min
Ollama progress: 50/11923 | checkpoint saved | ETA ~ 1083.9 min
Ollama progress: 75/11923 | checkpoint saved | ETA ~ 1075.7 min
Ollama progress: 100/11923 | checkpoint saved | ETA ~ 1102.7 min
Ollama progress: 125/11923 | checkpoint saved | ETA ~ 1120.7 min
Ollama progress: 150/11923 | checkpoint saved | ETA ~ 1130.2 min
Ollama progress: 175/11923 | checkpoint saved | ETA ~ 1142.3 min
Ollama progress: 200/11923 | checkpoint saved | ETA ~ 1140.6 min
Ollama progress: 225/11923 | checkpoint saved | ETA ~ 1143.3 min
Ollama progress: 250/11923 | checkpoint saved | ETA ~ 1150.0 min
Ollama progress: 275/11923 | checkpoint saved | ETA ~ 1156.7 min
Ollama progress: 300/11923 | checkpoint saved | ETA ~ 1155.2 min
Ollama progress: 325/11923 | checkpoint saved | ETA ~ 1146.6 min
Ollama progress: 350/11923 | checkpoint saved | ETA ~ 1140.9 min
Ollama progress: 375/11923 | checkpoint saved | ETA ~ 1133.7 min
Ollama progress: 400/11923 |

,tweet,y_test_label,ollama_prediction,is_match
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False
1,typa girl bullied lesbians high school,age,gender,False
2,cool woman gave directions polling place whose...,age,gender,False
3,need teach version quran terrorists killing pe...,religion,religion,True
4,rome destroy coliseum roman forum every buildi...,religion,religion,True
5,try hide jihadi propaganda know hell gys specl...,religion,religion,True
6,city college,not_cyberbullying,not_cyberbullying,True
7,pepperidge farm remembers,other_cyberbullying,not_cyberbullying,False
8,added feminazi white knight professional victi...,other_cyberbullying,gender,False
9,blm something know muslim leaders record stati...,religion,religion,True


Done. Collected 11923 predictions from model: qwen2.5:7b-instruct
Total exact label matches: 5277/11923
Checkpoint file: Artifacts\ollama_checkpoint_qwen2.5_7b-instruct.pkl
Predictions artifact: Artifacts\ollama_predictions_qwen2.5_7b-instruct.pkl


#### **Evaluation**

In [18]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Ensure y_eval is integer index array and lengths match predictions.
y_eval_idx = np.asarray(y_eval).astype(int)
ollama_pred_idx = np.asarray(ollama_pred_ids).astype(int)

if len(y_eval_idx) != len(ollama_pred_idx):
    raise ValueError(
        f"Length mismatch: y_eval has {len(y_eval_idx)} rows, ollama_pred has {len(ollama_pred_idx)} rows"
    )

# Keep readable labels and index columns for inspection.
ollama_pred_df["true_label_idx"] = y_eval_idx
ollama_pred_df["ollama_pred_idx"] = ollama_pred_idx
display(ollama_pred_df.head(10))

p, r, f1, support = precision_recall_fscore_support(
    y_eval_idx,
    ollama_pred_idx,
    average="weighted",
    zero_division=0,
 )

ollama_metrics = {
    "Accuracy": accuracy_score(y_eval_idx, ollama_pred_idx),
    "Precision": p,
    "Recall": r,
    "F1-Score": f1,
    "Support": int(len(y_eval_idx)),
}

ollama_metrics_df = pd.DataFrame(ollama_metrics, index=[OLLAMA_MODEL]).T
display(ollama_metrics_df)

print("\nLLM classification report:\n")
print(
    classification_report(
        y_eval_idx,
        ollama_pred_idx,
        target_names=class_names_list,
        zero_division=0,
    )
)

,tweet,y_test_label,ollama_prediction,is_match,true_label_idx,ollama_pred_idx
0,today tomorrow try move mountains cross fingers,other_cyberbullying,not_cyberbullying,False,4,3
1,typa girl bullied lesbians high school,age,gender,False,0,2
2,cool woman gave directions polling place whose...,age,gender,False,0,2
3,need teach version quran terrorists killing pe...,religion,religion,True,5,5
4,rome destroy coliseum roman forum every buildi...,religion,religion,True,5,5
5,try hide jihadi propaganda know hell gys specl...,religion,religion,True,5,5
6,city college,not_cyberbullying,not_cyberbullying,True,3,3
7,pepperidge farm remembers,other_cyberbullying,not_cyberbullying,False,4,3
8,added feminazi white knight professional victi...,other_cyberbullying,gender,False,4,2
9,blm something know muslim leaders record stati...,religion,religion,True,5,5


,qwen2.5:7b-instruct
Accuracy,0.442590
Precision,0.524460
Recall,0.442590
F1-Score,0.428401
Support,11923.000000



LLM classification report:

                     precision    recall  f1-score   support

                age       0.51      0.02      0.04      1952
          ethnicity       0.79      0.42      0.55      1995
             gender       0.48      0.44      0.46      1952
  not_cyberbullying       0.39      0.69      0.50      2040
other_cyberbullying       0.13      0.23      0.17      2006
           religion       0.85      0.85      0.85      1978

           accuracy                           0.44     11923
          macro avg       0.53      0.44      0.43     11923
       weighted avg       0.52      0.44      0.43     11923



### **Zero-shot NLI (BART-MNLI)**

#### **Predictions**

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from transformers import pipeline
import torch
import re
import time
import joblib

try:
    from datasets import Dataset
    from transformers.pipelines.pt_utils import KeyDataset
except ImportError as e:
    raise ImportError(
        "datasets is required for efficient GPU batching. Install with: pip install datasets"
    ) from e

NLI_MODEL = "facebook/bart-large-mnli"
NLI_BATCH_SIZE = 16
NLI_HYPOTHESIS_TEMPLATE = "This tweet belongs to the {} category."

# Checkpoint/resume configuration (same concept as Ollama)
NLI_CHECKPOINT_EVERY = 100
NLI_RESUME_FROM_CHECKPOINT = True
NLI_RESET_CHECKPOINT = False

# Use CUDA when available; otherwise use CPU.
HF_DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Zero-shot device: {'cuda' if HF_DEVICE == 0 else 'cpu'}")

# Reuse same eval split as Ollama section when available.
if "x_eval" not in globals() or "y_eval" not in globals():
    x_eval = list(x_test)
    y_eval = np.asarray(y_test)

if "class_names_list" not in globals():
    class_names_list = [str(x) for x in list(class_names)]

label_to_id = {label: i for i, label in enumerate(class_names_list)}
candidate_labels = list(class_names_list)

if len(candidate_labels) == 0:
    raise ValueError("candidate_labels is empty. Please ensure class_names is loaded.")

# Artifact paths for zero-shot NLI
artifacts_dir = Path(r"/content/drive/MyDrive/Colab Notebooks/CSCI 4907 NLU/Project/Code/Artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)
safe_nli_name = re.sub(r"[^a-zA-Z0-9_.-]", "_", NLI_MODEL)
nli_checkpoint_path = artifacts_dir / f"nli_checkpoint_{safe_nli_name}.pkl"
nli_predictions_path = artifacts_dir / f"nli_predictions_{safe_nli_name}.pkl"

if NLI_RESET_CHECKPOINT and nli_checkpoint_path.exists():
    nli_checkpoint_path.unlink()
    print(f"Deleted old NLI checkpoint: {nli_checkpoint_path}")

def _sanitize_tweet_text(t) -> str:
    text = "" if t is None else str(t)
    text = text.strip()
    # Zero-shot pipeline can raise ValueError when a sequence is empty.
    return text if text else "[EMPTY_TWEET]"

# Prepare text list once; sanitize empty/whitespace-only values.
x_eval_text = [_sanitize_tweet_text(t) for t in x_eval]
total = len(x_eval_text)
if total == 0:
    raise ValueError("x_eval has no samples to predict.")

# Resume state
nli_pred_labels = []
start_index = 0

if NLI_RESUME_FROM_CHECKPOINT and nli_checkpoint_path.exists():
    nli_state = joblib.load(nli_checkpoint_path)
    same_model = nli_state.get("model") == NLI_MODEL
    same_size = int(nli_state.get("x_eval_size", -1)) == total
    same_labels = nli_state.get("candidate_labels", []) == candidate_labels
    if same_model and same_size and same_labels:
        nli_pred_labels = list(nli_state.get("pred_labels", []))
        start_index = int(nli_state.get("next_index", len(nli_pred_labels)))
        start_index = max(0, min(start_index, total))
        print(f"Resumed NLI checkpoint: {nli_checkpoint_path}")
        print(f"Loaded {len(nli_pred_labels)} saved NLI predictions; continuing from index {start_index}")
    else:
        print("NLI checkpoint found but skipped due to mismatch (model/test-size/labels).")

def save_nli_checkpoint(next_index: int, pred_labels):
    state = {
        "model": NLI_MODEL,
        "candidate_labels": candidate_labels,
        "x_eval_size": total,
        "next_index": int(next_index),
        "pred_labels": list(pred_labels),
        "saved_at_unix": time.time(),
    }
    joblib.dump(state, nli_checkpoint_path)

# Initialize classifier once.
nli_classifier = pipeline(
    task="zero-shot-classification",
    model=NLI_MODEL,
    device=HF_DEVICE,
 )

# Dataset-based pipeline input for better GPU utilization and stability
if start_index < total:
    run_dataset = Dataset.from_dict({"text": x_eval_text[start_index:]})
    keyed = KeyDataset(run_dataset, "text")

    start_time = time.time()
    for local_i, out in enumerate(
        nli_classifier(
            keyed,
            candidate_labels=candidate_labels,
            multi_label=False,
            hypothesis_template=NLI_HYPOTHESIS_TEMPLATE,
            batch_size=NLI_BATCH_SIZE,
            truncation=True,
        ),
        start=1,
    ):
        pred_label = out["labels"][0]
        nli_pred_labels.append(pred_label)

        processed = start_index + local_i
        if (processed % NLI_CHECKPOINT_EVERY == 0) or (processed == total):
            save_nli_checkpoint(next_index=processed, pred_labels=nli_pred_labels)
            elapsed = time.time() - start_time
            done_now = max(1, processed - start_index)
            sec_per_tweet = elapsed / done_now
            remaining = total - processed
            eta_min = (remaining * sec_per_tweet) / 60.0
            print(f"Zero-shot NLI progress: {processed}/{total} | checkpoint saved | ETA ~ {eta_min:.1f} min")

if len(nli_pred_labels) != total:
    raise RuntimeError(
        f"NLI predictions incomplete: expected {total}, got {len(nli_pred_labels)}"
    )

nli_pred_ids = np.asarray([label_to_id[label] for label in nli_pred_labels], dtype=int)
y_eval_idx = np.asarray(y_eval).astype(int)

if len(y_eval_idx) != len(nli_pred_ids):
    raise ValueError(
        f"Length mismatch: y_eval has {len(y_eval_idx)} rows, nli_pred has {len(nli_pred_ids)} rows"
    )

nli_pred_df = pd.DataFrame({
    "tweet": x_eval_text,
    "y_test_label": [class_names_list[int(i)] for i in y_eval_idx],
    "nli_prediction": nli_pred_labels,
})
nli_pred_df["is_match"] = nli_pred_df["y_test_label"] == nli_pred_df["nli_prediction"]

# Save final NLI predictions artifact
joblib.dump(
    {
        "nli_pred_ids": nli_pred_ids,
        "nli_pred_labels": nli_pred_labels,
        "nli_pred_df": nli_pred_df,
        "model": NLI_MODEL,
        "candidate_labels": candidate_labels,
        "x_eval_size": total,
    },
    nli_predictions_path,
 )

display(nli_pred_df.head(20))
print(f"Done. Collected {len(nli_pred_ids)} predictions from model: {NLI_MODEL}")
print(f"NLI checkpoint file: {nli_checkpoint_path}")
print(f"NLI predictions artifact: {nli_predictions_path}")

Zero-shot device: cuda
Resumed NLI checkpoint: /content/drive/MyDrive/Colab Notebooks/CSCI 4907 NLU/Project/Code/Artifacts/nli_checkpoint_facebook_bart-large-mnli.pkl
Loaded 1800 saved NLI predictions; continuing from index 1800


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Zero-shot NLI progress: 1900/11923 | checkpoint saved | ETA ~ 9.3 min
Zero-shot NLI progress: 2000/11923 | checkpoint saved | ETA ~ 9.9 min
Zero-shot NLI progress: 2100/11923 | checkpoint saved | ETA ~ 9.6 min
Zero-shot NLI progress: 2200/11923 | checkpoint saved | ETA ~ 9.4 min
Zero-shot NLI progress: 2300/11923 | checkpoint saved | ETA ~ 9.4 min
Zero-shot NLI progress: 2400/11923 | checkpoint saved | ETA ~ 9.3 min
Zero-shot NLI progress: 2500/11923 | checkpoint saved | ETA ~ 9.2 min
Zero-shot NLI progress: 2600/11923 | checkpoint saved | ETA ~ 9.1 min
Zero-shot NLI progress: 2700/11923 | checkpoint saved | ETA ~ 9.0 min
Zero-shot NLI progress: 2800/11923 | checkpoint saved | ETA ~ 9.0 min
Zero-shot NLI progress: 2900/11923 | checkpoint saved | ETA ~ 9.0 min
Zero-shot NLI progress: 3000/11923 | checkpoint saved | ETA ~ 8.8 min
Zero-shot NLI progress: 3100/11923 | checkpoint saved | ETA ~ 8.8 min
Zero-shot NLI progress: 3200/11923 | checkpoint saved | ETA ~ 8.7 min
Zero-shot NLI progre

,tweet,y_test_label,nli_prediction,is_match
0,today tomorrow try move mountains cross fingers,other_cyberbullying,age,False
1,typa girl bullied lesbians high school,age,age,True
2,cool woman gave directions polling place whose...,age,age,True
3,need teach version quran terrorists killing pe...,religion,religion,True
4,rome destroy coliseum roman forum every buildi...,religion,religion,True
5,try hide jihadi propaganda know hell gys specl...,religion,ethnicity,False
6,city college,not_cyberbullying,age,False
7,pepperidge farm remembers,other_cyberbullying,ethnicity,False
8,added feminazi white knight professional victi...,other_cyberbullying,gender,False
9,blm something know muslim leaders record stati...,religion,religion,True


Done. Collected 11923 predictions from model: facebook/bart-large-mnli
NLI checkpoint file: /content/drive/MyDrive/Colab Notebooks/CSCI 4907 NLU/Project/Code/Artifacts/nli_checkpoint_facebook_bart-large-mnli.pkl
NLI predictions artifact: /content/drive/MyDrive/Colab Notebooks/CSCI 4907 NLU/Project/Code/Artifacts/nli_predictions_facebook_bart-large-mnli.pkl


#### **Evaluation**

In [ ]:
p, r, f1, support = precision_recall_fscore_support(
    y_eval_idx,
    nli_pred_ids,
    average="weighted",
    zero_division=0,
 )

nli_metrics = {
    "Accuracy": accuracy_score(y_eval_idx, nli_pred_ids),
    "Precision": p,
    "Recall": r,
    "F1-Score": f1,
    "Support": int(len(y_eval_idx)),
}

nli_metrics_df = pd.DataFrame(nli_metrics, index=[NLI_MODEL]).T
display(nli_metrics_df)

print("\nZero-shot NLI classification report:\n")
print(
    classification_report(
        y_eval_idx,
        nli_pred_ids,
        target_names=class_names_list,
        zero_division=0,
    )
)

,facebook/bart-large-mnli
Accuracy,0.446532
Precision,0.450896
Recall,0.446532
F1-Score,0.399407
Support,11923.000000



Zero-shot NLI classification report:

                     precision    recall  f1-score   support

                age       0.35      0.61      0.44      1952
          ethnicity       0.52      0.83      0.64      1995
             gender       0.38      0.63      0.48      1952
  not_cyberbullying       0.27      0.08      0.12      2040
other_cyberbullying       0.34      0.07      0.11      2006
           religion       0.86      0.49      0.62      1978

           accuracy                           0.45     11923
          macro avg       0.45      0.45      0.40     11923
       weighted avg       0.45      0.45      0.40     11923



### **OpenAI**

In [ ]:
import os
import json
import re
import time
import numpy as np
import pandas as pd
import joblib
import torch
from openai import OpenAI
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# ----------------------------
# Configuration
# ----------------------------
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
MAX_OPENAI_CASES = None  # e.g., 200 for quick test; None uses all tweets in x_test split
REQUEST_TIMEOUT_SECONDS = 45
MAX_RETRIES = 3

# ----------------------------
# OpenAI API setup
# ----------------------------
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. PowerShell example: $env:OPENAI_API_KEY='your_key_here'"
    )

client = OpenAI(api_key=api_key, timeout=REQUEST_TIMEOUT_SECONDS)

SYSTEM_PROMPT = f"""You are an expert annotator for cyberbullying tweet classification.

Given the labels provided: {class_names}

Classify the tweet provided to you to the closest matching label (cyberbullying category).

Return strictly just one output label from {class_names}, i.e. "age"
"""

def parse_json_payload(raw_text):
    text = (raw_text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE | re.DOTALL).strip()
    return json.loads(text)

def openai_predict_one(tweet_text):
    user_prompt = f"Classify this tweet:\n\n{tweet_text}"
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )
            content = resp.choices[0].message.content
            payload = parse_json_payload(content)
            label = str(payload.get("label", "")).strip()
            if label not in label_to_id:
                raise ValueError(f"Invalid label returned: {label}")
            reason = str(payload.get("reason", "")).strip()
            return label_to_id[label], label, reason
        except Exception as err:
            last_error = err
            if attempt < MAX_RETRIES:
                time.sleep(1.2 * attempt)

    raise RuntimeError(f"OpenAI prediction failed after retries: {last_error}")

# ----------------------------
# Run OpenAI prediction for each tweet
# ----------------------------
openai_pred_ids = []
openai_rows = []

total = len(x_eval)
for i, tweet_text in enumerate(x_eval, start=1):
    pred_id, pred_label, reason = openai_predict_one(str(tweet_text))
    openai_pred_ids.append(pred_id)
    openai_rows.append({
        "tweet": str(tweet_text),
        "true_label": class_names_cls[int(y_eval[i - 1])],
        "openai_pred": pred_label,
        "reason": reason,
    })
    if (i % 25 == 0) or (i == total):
        print(f"OpenAI progress: {i}/{total}")

openai_pred_ids = np.asarray(openai_pred_ids)
openai_pred_df = pd.DataFrame(openai_rows)
display(openai_pred_df.head(10))

# ----------------------------
# Compute evaluation metrics
# ----------------------------
def metric_summary(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": p,
        "Recall": r,
        "F1-Score": f1,
        "Support": len(y_true),
    }

summary_map = {}
for model_name, y_pred in baseline_preds.items():
    summary_map[model_name] = metric_summary(y_eval, y_pred)

summary_map["OpenAI"] = metric_summary(y_eval, openai_pred_ids)
if bilstm_summary is not None:
    summary_map["Bi-LSTM"] = bilstm_summary

col_order = [
    "Naive Bayes",
    "Logistic Regression",
    "SVM",
    "Random Forest",
    "Bi-LSTM",
    "OpenAI",
]
col_order = [c for c in col_order if c in summary_map]

comparison_df = pd.DataFrame({c: summary_map[c] for c in col_order})
comparison_df = comparison_df.loc[["Accuracy", "Precision", "Recall", "F1-Score", "Support"]]
display(comparison_df)

print("\nOpenAI classification report:\n")
print(classification_report(y_eval, openai_pred_ids, target_names=class_names_cls, zero_division=0))

# LaTeX output for updating your markdown comparison table
latex_df = comparison_df.copy()
for metric_name in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    if metric_name in latex_df.index:
        latex_df.loc[metric_name] = [f"{float(v):.3f}" for v in latex_df.loc[metric_name].values]
if "Support" in latex_df.index:
    latex_df.loc["Support"] = [str(int(float(v))) for v in latex_df.loc["Support"].values]

print("\nLaTeX-friendly table with OpenAI column:\n")
print(latex_df.to_latex())